# Autoresearch Experiment Analysis

Analysis of autonomous EMR model tuning results from `results.tsv`.

**Metrics** (all evaluated on the held-out val set via autoregressive generation — see `evaluation.py`):
- `outcome_auroc` — mean per-complication AUROC from pooled episode-level AUC (**primary**, ↑)
- `outcome_auprc` — mean per-complication AUPRC from the same evaluation (**secondary**, ↑)
- `onset_mae_hrs` — mean absolute error in predicted onset time in hours (**tertiary**, ↓)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

df = pd.read_csv("results.tsv", sep="\t")
df["outcome_auroc"] = pd.to_numeric(df["outcome_auroc"], errors="coerce")
df["outcome_auprc"] = pd.to_numeric(df["outcome_auprc"], errors="coerce")
df["onset_mae_hrs"] = pd.to_numeric(df["onset_mae_hrs"], errors="coerce")
df["peak_vram_gb"]  = pd.to_numeric(df["peak_vram_gb"],  errors="coerce")
df["status"]        = df["status"].str.strip().str.upper()

print(f"Total experiments: {len(df)}")
print(f"Columns: {list(df.columns)}")
df.head(10)


In [2]:
counts = df["status"].value_counts()
print("Experiment outcomes:")
print(counts.to_string())

n_keep    = counts.get("KEEP",    0)
n_discard = counts.get("DISCARD", 0)
n_crash   = counts.get("CRASH",   0)
n_decided = n_keep + n_discard
if n_decided > 0:
    print(f"\nKeep rate: {n_keep}/{n_decided} = {n_keep / n_decided:.1%}")

Experiment outcomes:
status
DISCARD     41
KEEP        17
CRASH        1
BASELINE     1

Keep rate: 17/58 = 29.3%


In [ ]:
kept = df[df["status"] == "KEEP"].copy()
print(f"KEPT experiments ({len(kept)} total):\n")
for i, row in kept.iterrows():
    auroc = row["outcome_auroc"]
    auprc = row["outcome_auprc"]
    mae   = row["onset_mae_hrs"]
    desc  = row["description"]
    print(f"  #{i:3d}  auroc={auroc:.4f}  auprc={auprc:.4f}  mae={mae:.1f}h  mem={row['peak_vram_gb']:.1f}GB  {desc}")


## Metrics Over Experiments

Three panels:
1. **Outcome AUROC** — episode-level mean AUROC across all complications (primary, ↑)
2. **Outcome AUPRC** — episode-level mean AUPRC across all complications (secondary, ↑)
3. **Onset MAE** — mean absolute error between predicted and actual complication onset in hours (tertiary, ↓)

All metrics are computed on the held-out val set via autoregressive generation (see `evaluation.py`).
0.5 AUROC = random baseline.


In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 15), sharex=True)

valid     = df[df["status"] != "CRASH"].copy().reset_index(drop=True)
kept_mask = valid["status"] == "KEEP"
kept_idx  = valid.index[kept_mask]
kept_v    = valid[kept_mask]
disc      = valid[valid["status"] == "DISCARD"]

def _annotate(ax, kept_idx, kept_v, col):
    for idx, val in zip(kept_idx, kept_v[col]):
        if pd.isna(val):
            continue
        desc = str(valid.loc[idx, "description"]).strip()
        if len(desc) > 45:
            desc = desc[:42] + "..."
        ax.annotate(desc, (idx, val), textcoords="offset points", xytext=(6, 6),
                    fontsize=8, color="#1a7a3a", alpha=0.9, rotation=30, ha="left", va="bottom")

# ── Panel 1: Outcome AUROC (primary) ────────────────────────────────────────
ax = axes[0]
ax.axhline(0.5, color="#aaaaaa", linewidth=1, linestyle="--", label="Random (0.5)")
ax.scatter(disc.index, disc["outcome_auroc"], c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")
ax.scatter(kept_v.index, kept_v["outcome_auroc"], c="#2ecc71", s=50, zorder=4,
           label="Kept", edgecolors="black", linewidths=0.5)
running_best = kept_v["outcome_auroc"].cummax()
ax.step(kept_idx, running_best, where="post", color="#27ae60", linewidth=2, alpha=0.7, zorder=3, label="Running best")
_annotate(ax, kept_idx, kept_v, "outcome_auroc")
ax.set_ylabel("Outcome AUROC ↑", fontsize=11)
ax.set_title(f"Autoresearch EMR Progress — {len(df)} Experiments, {kept_mask.sum()} Kept", fontsize=13)
ax.legend(loc="lower right", fontsize=9)
ax.grid(True, alpha=0.2)

# ── Panel 2: Outcome AUPRC (secondary) ──────────────────────────────────────
ax2 = axes[1]
ax2.scatter(disc.index, disc["outcome_auprc"], c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")
ax2.scatter(kept_v.index, kept_v["outcome_auprc"], c="#9b59b6", s=50, zorder=4,
            label="Kept", edgecolors="black", linewidths=0.5)
running_best_auprc = kept_v["outcome_auprc"].cummax()
ax2.step(kept_idx, running_best_auprc, where="post", color="#8e44ad", linewidth=2, alpha=0.7, zorder=3, label="Running best")
ax2.set_ylabel("Outcome AUPRC ↑", fontsize=11)
ax2.legend(loc="lower right", fontsize=9)
ax2.grid(True, alpha=0.2)

# ── Panel 3: Onset MAE (tertiary) ────────────────────────────────────────────
ax3 = axes[2]
ax3.scatter(disc.index, disc["onset_mae_hrs"], c="#cccccc", s=12, alpha=0.5, zorder=2, label="Discarded")
ax3.scatter(kept_v.index, kept_v["onset_mae_hrs"], c="#e67e22", s=50, zorder=4,
            label="Kept", edgecolors="black", linewidths=0.5)
running_best_mae = kept_v["onset_mae_hrs"].cummin()
ax3.step(kept_idx, running_best_mae, where="post", color="#d35400", linewidth=2, alpha=0.7, zorder=3, label="Running best")
ax3.set_xlabel("Experiment #", fontsize=11)
ax3.set_ylabel("Onset MAE (hours) ↓", fontsize=11)
ax3.legend(loc="upper right", fontsize=9)
ax3.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()


## Summary Statistics

In [ ]:
kept           = df[df["status"] == "KEEP"].copy()
baseline_auroc = df.iloc[0]["outcome_auroc"]
baseline_auprc = df.iloc[0]["outcome_auprc"]
baseline_mae   = df.iloc[0]["onset_mae_hrs"]
best_auroc     = kept["outcome_auroc"].max()
best_row       = kept.loc[kept["outcome_auroc"].idxmax()]

print(f"Baseline outcome_auroc:  {baseline_auroc:.6f}")
print(f"Best outcome_auroc:      {best_auroc:.6f}")
print(f"Total improvement:       {best_auroc - baseline_auroc:+.6f}  "
      f"({(best_auroc - baseline_auroc) / max(baseline_auroc, 1e-9) * 100:.2f}%)")
print(f"Best experiment:         {best_row['description']}")
print()
print(f"Baseline outcome_auprc:  {baseline_auprc:.6f}")
print(f"Best outcome_auprc:      {kept['outcome_auprc'].max():.6f}")
print()
print(f"Baseline onset_mae_hrs:  {baseline_mae:.2f}")
print(f"Best onset_mae_hrs:      {kept['onset_mae_hrs'].min():.2f}")
print()
print("Cumulative effort per improvement (KEEP experiments in order):")
kept_sorted = kept.reset_index()
for _, row in kept_sorted.iterrows():
    desc = str(row["description"]).strip()
    print(f"  #{row['index']:3d}  auroc={row['outcome_auroc']:.4f}  "
          f"auprc={row['outcome_auprc']:.4f}  mae={row['onset_mae_hrs']:.1f}h  {desc}")


## Top Hits (Kept Experiments by Improvement)

In [ ]:
kept = df[df["status"] == "KEEP"].copy()
kept["prev_auroc"] = kept["outcome_auroc"].shift(1)
kept["delta"]      = kept["outcome_auroc"] - kept["prev_auroc"]

hits = kept.iloc[1:].copy().sort_values("delta", ascending=False)

print(f"{'Rank':>4}  {'Delta AUROC':>11}  {'AUROC':>7}  {'AUPRC':>7}  {'MAE(h)':>7}  Description")
print("-" * 90)
for rank, (_, row) in enumerate(hits.iterrows(), 1):
    print(f"{rank:4d}  {row['delta']:+.6f}   {row['outcome_auroc']:.4f}  "
          f"{row['outcome_auprc']:.4f}  {row['onset_mae_hrs']:>7.1f}  {row['description']}")

print(f"\n{'':>4}  {hits['delta'].sum():+.6f}   {'':>7}  {'':>7}  {'':>7}  TOTAL AUROC improvement over baseline")
